# Convert h5ad to Loupe Browser (.cloupe) files

Converts the final labelled AnnData objects for each timepoint into `.cloupe` files
for viewing in 10x Genomics Loupe Browser.

**Kernel:** `loupe-env`  (Python 3.11, loupepy 1.1.1)

**Datasets converted:**
- D2 Dz, D2 Lapa, D4 AS, D4 Dz, D4 Lapa, D10 Lapa, G6
- D10 Lapa EEC subset with kNN-refined labels (from notebook 1.4)

## 1. Imports and config

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../.." ).resolve()))

from src.config import *
from src.loupe import convert_h5ad_to_cloupe, ensure_converter_symlink

## 2. First-time setup

Run **once** to accept the 10x EULA, download the converter binary,
and create a space-free symlink. Type **yes** when prompted.

In [ ]:
from loupepy import setup
setup()
print(f"Converter: {ensure_converter_symlink()}")

## 3. Output directory

In [ ]:
LOUPE_DIR = ANALYSIS_DIR / "loupe"
LOUPE_DIR.mkdir(exist_ok=True)
print(f"Output: {LOUPE_DIR}")

## 4. Convert all timepoint datasets

In [ ]:
for name, info in DATASETS.items():
    h5ad_path = info["labelled"]
    out_path = LOUPE_DIR / f"{name}.cloupe"

    if not h5ad_path.exists():
        print(f"SKIP  {name} — file not found: {h5ad_path.name}")
        continue
    if out_path.exists():
        print(f"SKIP  {name} — already exists ({out_path.stat().st_size/1e6:.0f} MB)")
        continue

    print(f"\n{'='*60}")
    print(f"Converting {name}")
    print(f"{'='*60}")
    convert_h5ad_to_cloupe(h5ad_path, out_path)

## 5. Convert D10 Lapa EEC subset (kNN-refined labels)

In [ ]:
eec_path = MANUAL_LABELLED_2_DIR / "knn_EECs_egfDuod_D10_Lapa_DZ.h5ad"
eec_out = LOUPE_DIR / "D10_Lapa_EEC_subset.cloupe"

if eec_out.exists():
    print(f"Removing old EEC subset cloupe to regenerate...")
    eec_out.unlink()

print("Converting D10 Lapa EEC subset (kNN labels)")
convert_h5ad_to_cloupe(eec_path, eec_out)

## 6. Summary

In [ ]:
print("Generated .cloupe files:")
for f in sorted(LOUPE_DIR.glob("*.cloupe")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s}  {size_mb:>8.1f} MB")